# Klein unified training
This notebook controls the same sync, discovery, queue, and immutable-Hub report refresh code used by `/run-unified-training`. Choose one explicit action, then **Run All**. The default `status` action only reads durable state and does not clone code or request a token.

In [ ]:
SOURCE_REVISION = "24f4481999c4923b4e0deb17902ee6466a8234a4"
SOURCE_CHECKOUT = "/tmp/ai-toolkit-perceptual-pinned"
TRAINER_ROOT = "/app/ai-toolkit"  # validated native trainer inside the Alternative Training image
REPO_ID = "daverave/Personal"
RUN_ID = "e372619d-f4dc-4f4b-a2ba-b0df127e24fc"  # editable, never guessed
DATASET_ROOT = "/storage/datasets"
COMFYUI_ROOT = "/storage/ComfyUI"
LORA_ROOT = "/storage/ComfyUI/models/loras"
WORK_ROOT = "/storage/automation/unified"
WORKER_ID, WORKER_COUNT = 0, 1
RANKS = [1]
MODEL_IDS = []  # empty means every model; historical fallback below covers the original six
HISTORICAL_MODEL_IDS = [1, 2, 3, 4, 5, 6]
ARCHIVE_JOB_IDS = []  # empty refreshes every completed job in RUN_ID
ORDER = []      # normalized folder or catalog names; unlisted jobs stay alphabetical
ACTION = "status"  # status | sync | discover | run | refresh

In [ ]:
import json, os, pathlib, shutil, subprocess, sys
if ACTION == 'status':
    status_path = pathlib.Path(WORK_ROOT) / 'supervisor-state.json'
    result = json.loads(status_path.read_text(encoding='utf-8')) if status_path.is_file() else {'schema_version': 1, 'status': 'not-started'}
else:
    checkout = pathlib.Path(SOURCE_CHECKOUT)
    if checkout.exists(): shutil.rmtree(checkout)
    subprocess.run(['git', 'clone', '--filter=blob:none', 'https://github.com/Explyy/ai-toolkit-perceptual.git', str(checkout)], check=True)
    subprocess.run(['git', '-C', str(checkout), 'checkout', '--detach', SOURCE_REVISION], check=True)
    actual_revision = subprocess.run(['git', '-C', str(checkout), 'rev-parse', 'HEAD'], check=True, capture_output=True, text=True).stdout.strip()
    assert actual_revision == SOURCE_REVISION
    subprocess.run([sys.executable, '-m', 'pip', 'install', '--quiet', 'huggingface_hub>=0.27,<2', 'PyYAML>=6,<7', 'Pillow>=12,<13', 'numpy>=1.26,<3'], check=True)
    sys.path.insert(0, str(checkout))
    import getpass, yaml
    token = os.environ.get('HF_TOKEN') or getpass.getpass('HF private repository token: ')
    os.environ['HF_TOKEN'] = token
    os.environ['HF_REPO_ID'] = REPO_ID
    os.environ['DATASETS_FOLDER'] = DATASET_ROOT
    config = {
     'schema_version': 1, 'dataset_root': DATASET_ROOT, 'comfyui_root': COMFYUI_ROOT, 'loras_root': LORA_ROOT, 'work_root': WORK_ROOT,
     'gui_database': str(pathlib.Path(TRAINER_ROOT) / 'aitk_db.db'), 'initialize_gui_dataset_root': True,
     'hub': {'repo_id': REPO_ID, 'repo_type': 'dataset', 'token_env': 'HF_TOKEN'},
     'worker': {'id': WORKER_ID, 'count': WORKER_COUNT},
     'sync': {'catalog_prefix': 'training-backups', 'results_prefix': 'training-results', 'legacy_runs': [{'run_id': RUN_ID, 'model_ids': HISTORICAL_MODEL_IDS}]},
     'discovery': {'quiet_seconds': 60, 'target_exposures': 126, 'ledger_path': 'training-automation/workflow-ledger.json', 'retry_incomplete': False, 'order': ORDER},
     'queue': {'trainer_yaml': str(checkout / 'config/examples/klein_automation/trainer-subject-likeness-masked-klein-9b-v2.yaml'), 'repo_root': TRAINER_ROOT, 'output_root': '/storage/output', 'trigger_word': 'Owhx'},
     'checkpoint_policy': {'save_every': 100, 'max_local_step_saves': 5},
     'evaluation': {'enabled': True, 'require_identity_available': True, 'reference_provenance': 'training-set', 'face_backend': 'training_automation.backends:InsightFaceCPUBackend', 'face_backend_options': {'model_dir': '/opt/training-automation-models/insightface/models/buffalo_l'}, 'reference_identity_filter': {'single_face_only': True, 'minimum_valid_count': 3, 'minimum_valid_fraction': 0.5}, 'landmark_backend': 'training_automation.backends:UltralyticsPoseCPUBackend', 'landmark_backend_options': {'model_path': '/opt/training-automation-models/ultralytics/yolo11n-pose.pt', 'expected_sha256': '869e83fcdffdc7371fa4e34cd8e51c838cc729571d1635e5141e3075e9319dc0'}},
     'archive': {'remote_prefix': 'training-archives'}
    }
    config_path = pathlib.Path(WORK_ROOT) / 'notebook-config.yaml'
    config_path.parent.mkdir(parents=True, exist_ok=True)
    config_path.write_text(yaml.safe_dump(config, sort_keys=False), encoding='utf-8')
    from training_automation.backup import HuggingFaceBackupClient
    from training_automation.results import refresh_archived_run_reports
    from training_automation.unified import run_unified_workflow, sync_unified_loras
    client = HuggingFaceBackupClient(token)
    if ACTION == 'sync':
        result = sync_unified_loras(config_path, client=client, ranks=RANKS, model_ids=MODEL_IDS)
    elif ACTION in {'discover', 'run'}:
        result = run_unified_workflow(config_path, client=client, dry_run=(ACTION == 'discover'))
    elif ACTION == 'refresh':
        result = refresh_archived_run_reports(client=client, repo_id=REPO_ID, repo_type='dataset', run_id=RUN_ID, job_ids=ARCHIVE_JOB_IDS, work_dir=pathlib.Path(WORK_ROOT) / 'report-refresh')
    else:
        raise ValueError('ACTION must be status, sync, discover, run, or refresh')
print(json.dumps(result, indent=2, sort_keys=True))